# Libraries & Datasets

Lets start with importing the necessary libraries.

In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os

## Lets load our datasets

We need to remember that we will be working with the ASL dataset in `csv` format.

In [2]:
# We'll assume that the csv datasets are in the same folder than this notebook, and are
# named as follows:
# Train dataset: sign_mnist_train.csv
# Test or validation dataset: sign_mnist_valid.csv

data_path =  Path.cwd()
train_df = pd.read_csv(os.path.join(data_path, 'sign_mnist_train.csv'))
val_df  = pd.read_csv(os.path.join(data_path, 'sign_mnist_valid.csv'))


In [3]:
# Now, lets check what we got in our dataset...
train_df.head(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,3,107,118,127,134,139,143,146,150,153,...,207,207,207,207,206,206,206,204,203,202
1,6,155,157,156,156,156,157,156,158,158,...,69,149,128,87,94,163,175,103,135,149
2,2,187,188,188,187,187,186,187,188,187,...,202,201,200,199,198,199,198,195,194,195
3,2,211,211,212,212,211,210,211,210,210,...,235,234,233,231,230,226,225,222,229,163
4,12,164,167,170,172,176,179,180,184,185,...,92,105,105,108,133,163,157,163,164,179


... and after taking a quick peek, we can see that we already that we have our target column as `label` and a description of the grayscale intensity for each pixel of each image.

Then, we need to load our train dataset

In [4]:
# First, we load each array from the test and train sets
y_train = np.array(train_df['label'])
y_val   = np.array(val_df['label'])

# Then, we delete the `label` column from our datasets...
del train_df['label']
del val_df['label']

# ... so we can load only the pixel arrays easily into other variables
x_train = train_df.values.astype(np.float32)
x_val  = val_df.values.astype(np.float32)

# But, what about data leakage?

Because we're looking to minimize the phenomenom of data leakage, we should always look out for the way that we split our train dataset to validate the accuracy of our models... so, we now need to create our own split function to have the standard splitting practice:

- Train dataset
- Validation dataset
- Test dataset

In [10]:
def split_val_into_test(
        x: np.array,
        y:np.array,
        pct: float = 0.5,
        shuffle: bool =True):

    """  
    Parameters:

        - x: variables or inputs of the validation dataset (NumPy array).
        - y: Labels of the validation dataset (NumPy array).
        - pct: Proportion of the validation set to allocate to the test set (default is 0.5).
        - shuffle: Whether to shuffle the data before splitting (default is True).



    Returns:

        - x_val: Features for the validation set.
        - y_val: Labels for the validation set.
        - x_test: Features for the test set.
        - y_test: Labels for the test set. 
    """

    if shuffle:
        # Lets create some random indices to shuffle our dataset
        indices = np.arange(len(x))
        np.random.shuffle(indices)
        x = x[indices]
        y = y[indices]

    # Lets store for a little while the splitter percentage
    num_test_samples = int(len(x) * pct)

    # Now, we create our validation and test datasets
    x_test = x[:num_test_samples]
    y_test = y[:num_test_samples]

    x_val  = x[num_test_samples]
    y_val  = y[num_test_samples]

    return x_val, y_val, x_test, y_test

Now, lets split our validation dataset into a `val` and `test` sets

In [11]:
x_val, y_val, x_test, y_test = split_val_into_test(x_val, y_val)